# Full-Catalog Log-Likelihood Eval (KV-Cache Optimized)

**Status:** future work — not run as part of the original submission.

## What this does

Extends the shared-pool log-likelihood evaluation (03_2) to the **full ~6,117-item catalog**.
For each test user, scores every catalog title by mean per-token log-probability
of the title given the user's history, and produces a top-K ranked list.

Result is directly comparable to BPR-MF's full-catalog HR@10 = 0.0220 and the
fine-tuned generative title-generation HR@10 = 0.0266.

## Why a separate notebook

Naive scoring (the 03_2 method) recomputes the prompt forward pass for every candidate.
Over a 6,117-item catalog that's ~924 s/user on L4. This notebook reuses the prompt's
key-value cache so only the title tokens are processed per candidate (~12 tokens vs ~150),
cutting per-user cost to ~74 s on L4 / ~37 s on A100.

| Setup | Per user | Full 7,288 users |
|---|---|---|
| 03_2 method (L4) | ~924 s | ~78 days |
| **KV cache (L4)** | **~74 s** | **~6.2 days** |
| KV cache (A100) | ~37 s | ~3 days |

For a tractable approximation, see `02_full_catalog_two_stage.ipynb`.

## Requirements

- Colab A100 or L4
- QLoRA adapter from `03_1_genrec_qlora_finetuning.ipynb`
- `SMOKE_TEST = True` runs on 3 users to verify the pipeline before committing

## 0. Setup

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets torch

In [ ]:
SMOKE_TEST = True   # 3 users, no save — set False for the real run

if SMOKE_TEST:
    N_USERS = 3
    TOP_K_TO_SAVE = 50
    CHECKPOINT_EVERY = 1
    print('*** SMOKE TEST: 3 users, no save ***')
else:
    N_USERS = None
    TOP_K_TO_SAVE = 50
    CHECKPOINT_EVERY = 100
    print('*** FULL RUN ***')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = '/content/drive/MyDrive/Foundations of Large Language Models/Final Project'
DATA_DIR = f'{DRIVE_ROOT}/data'
RESULTS_DIR = f'{DRIVE_ROOT}/results'
ADAPTER_DIR = f'{DRIVE_ROOT}/results/genrec_qlora/final_adapter'

assert os.path.exists(f'{ADAPTER_DIR}/adapter_config.json'), \
    f'No adapter found at {ADAPTER_DIR}. Run notebook 03_1 first.'
print(f'Adapter found at {ADAPTER_DIR}')

## 1. Load data and model

In [ ]:
import json
import pickle
import numpy as np

with open(f'{DATA_DIR}/shared_data.pkl', 'rb') as f:
    shared = pickle.load(f)
with open(f'{DATA_DIR}/genrec_test.json') as f:
    genrec_test = json.load(f)

item_titles = shared['item_titles']
test_ground_truth = shared['test_ground_truth']

print(f'Test users: {len(test_ground_truth)}')
print(f'Catalog items: {len(item_titles)}')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto',
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

print(f'Model loaded with adapter. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 2. KV-cache scoring function

Forward-pass the prompt once and cache its `past_key_values`. Then for each candidate
title, run a small forward pass over the title tokens only, conditioned on the cached
prompt KV. The first title token is predicted by the last prompt logit (saved separately);
subsequent tokens use the title's own forward-pass logits.

We convert `past_key_values` to legacy tuple format via `to_legacy_cache()` so the same
reference can be reused across many calls without `DynamicCache` mutation issues.

In [ ]:
import torch.nn.functional as F


def score_titles_kvcache(prompt_text, titles):
    """Score N titles given a shared prompt, reusing the prompt KV.

    Returns a list of mean per-token log-probabilities, one per title.

    Implementation notes:
    - We tokenize (prompt + title) together and slice off the prompt-portion tokens.
      This matches the naive reference scorer exactly and avoids subtle BPE-merge-
      at-boundary discrepancies.
    - We pass explicit `position_ids` to the second forward pass. Without them, the
      model treats title tokens as positions 0,1,2,... instead of prompt_len, prompt_len+1, ...
      which causes large numerical drift from the reference scorer (verified locally —
      with position_ids, KV-cache and naive scores agree bit-exactly).
    """
    prompt_inputs = tokenizer(prompt_text, return_tensors='pt')
    prompt_ids = prompt_inputs['input_ids']
    prompt_len = prompt_ids.shape[1]

    with torch.no_grad():
        prompt_out = model(input_ids=prompt_ids.to(model.device), use_cache=True)
    prompt_kv = prompt_out.past_key_values
    if hasattr(prompt_kv, 'to_legacy_cache'):
        prompt_kv_legacy = prompt_kv.to_legacy_cache()
    else:
        prompt_kv_legacy = prompt_kv
    last_prompt_logit = prompt_out.logits[0, -1, :]

    scores = []
    for title in titles:
        if not title:
            scores.append(-float('inf'))
            continue
        full_ids = tokenizer(prompt_text + title, return_tensors='pt')['input_ids']
        if full_ids.shape[1] <= prompt_len:
            scores.append(-float('inf'))
            continue
        title_ids = full_ids[:, prompt_len:].to(model.device)
        title_len = title_ids.shape[1]

        position_ids = torch.arange(
            prompt_len, prompt_len + title_len, device=model.device
        ).unsqueeze(0)

        with torch.no_grad():
            title_out = model(
                input_ids=title_ids,
                past_key_values=prompt_kv_legacy,
                position_ids=position_ids,
                use_cache=False,
            )

        if title_len == 1:
            stacked = last_prompt_logit.unsqueeze(0)
        else:
            stacked = torch.cat(
                [last_prompt_logit.unsqueeze(0), title_out.logits[0, :-1, :]], dim=0
            )
        log_probs = F.log_softmax(stacked, dim=-1)
        token_lps = log_probs.gather(1, title_ids[0].unsqueeze(1)).squeeze(1)
        scores.append(token_lps.mean().item())

    return scores


In [ ]:
# Sanity check: ground truth should score higher than an off-topic title
sample = genrec_test[0]
prompt = f"{sample['instruction']}\n\n### input:\n{sample['input']}\n\n### Response:\n"

test_titles = [sample['output'], 'Cooking for Beginners: Easy Recipes']
scores = score_titles_kvcache(prompt, test_titles)
print(f'Ground truth: "{test_titles[0]}" → {scores[0]:.4f}')
print(f'Off-topic:    "{test_titles[1]}" → {scores[1]:.4f}')
assert scores[0] > scores[1], 'Sanity check failed: ground truth scored lower than off-topic'
print('Sanity check passed.')

In [ ]:
# Benchmark: score 100 catalog titles, project full-catalog cost
import time

bench_titles = list(item_titles.values())[:100]
_ = score_titles_kvcache(prompt, bench_titles[:5])  # warm up GPU

t0 = time.time()
_ = score_titles_kvcache(prompt, bench_titles)
elapsed = time.time() - t0

per_candidate_ms = elapsed / 100 * 1000
full_per_user_s = elapsed / 100 * len(item_titles)
print(f'100 candidates in {elapsed:.1f}s → {per_candidate_ms:.0f} ms/candidate')
print(f'Projected full-catalog ({len(item_titles)} items) per user: ~{full_per_user_s:.0f}s')
users = 7288 if N_USERS is None else N_USERS
print(f'Projected total for {users} users: ~{full_per_user_s * users / 3600:.1f} hours')

## 3. Run full-catalog scoring

For each user, score every catalog title and save the top-K. Resumable from checkpoint.

In [ ]:
test_by_user = {ex['user_idx']: ex for ex in genrec_test}

catalog_item_ids = sorted(item_titles.keys())
catalog_titles = [item_titles[i] for i in catalog_item_ids]
print(f'Catalog: {len(catalog_titles)} items')

eval_users = list(test_by_user.keys())
if N_USERS is not None:
    eval_users = eval_users[:N_USERS]
print(f'Users to evaluate: {len(eval_users)}')

In [ ]:
CHECKPOINT_PATH = f'{RESULTS_DIR}/genrec_full_catalog_kvcache_checkpoint.pkl'

if os.path.exists(CHECKPOINT_PATH) and not SMOKE_TEST:
    with open(CHECKPOINT_PATH, 'rb') as f:
        ckpt = pickle.load(f)
    full_predictions = ckpt['full_predictions']
    latencies = ckpt['latencies']
    print(f'Resumed from checkpoint: {len(full_predictions)} users done.')
else:
    full_predictions = {}
    latencies = []

remaining = [u for u in eval_users if u not in full_predictions]
print(f'Total: {len(eval_users)}, Remaining: {len(remaining)}')

for i, user_idx in enumerate(remaining):
    if user_idx not in test_by_user:
        continue

    example = test_by_user[user_idx]
    prompt = f"{example['instruction']}\n\n### input:\n{example['input']}\n\n### Response:\n"

    t0 = time.time()
    scores = score_titles_kvcache(prompt, catalog_titles)
    latencies.append(time.time() - t0)

    score_arr = np.array(scores)
    top_k_idx = np.argsort(score_arr)[::-1][:TOP_K_TO_SAVE]
    full_predictions[user_idx] = [catalog_item_ids[j] for j in top_k_idx]

    if (i + 1) % CHECKPOINT_EVERY == 0:
        recent = latencies[-CHECKPOINT_EVERY:]
        eta_h = np.mean(recent) * (len(eval_users) - len(full_predictions)) / 3600
        print(f'  {len(full_predictions)}/{len(eval_users)} | '
              f'latency: {np.mean(recent):.1f} s/user | '
              f'ETA: {eta_h:.1f} h')
        if not SMOKE_TEST:
            with open(CHECKPOINT_PATH, 'wb') as f:
                pickle.dump({'full_predictions': full_predictions, 'latencies': latencies}, f)

if not SMOKE_TEST:
    with open(CHECKPOINT_PATH, 'wb') as f:
        pickle.dump({'full_predictions': full_predictions, 'latencies': latencies}, f)

print(f'\nDone. {len(full_predictions)} users scored.')
if latencies:
    print(f'Mean latency: {np.mean(latencies):.1f} s/user')

## 4. Evaluate

In [ ]:
import math

def hit_at_k(ranked_list, ground_truth, k=10):
    return 1.0 if ground_truth in ranked_list[:k] else 0.0

def ndcg_at_k(ranked_list, ground_truth, k=10):
    for i, item in enumerate(ranked_list[:k]):
        if item == ground_truth:
            return 1.0 / math.log2(i + 2)
    return 0.0

def evaluate_ranking(preds, gt, k_values=None):
    if k_values is None:
        k_values = [5, 10, 20]
    results = {}
    for k in k_values:
        hits, ndcgs = [], []
        for uid, true_item in gt.items():
            if uid not in preds:
                continue
            hits.append(hit_at_k(preds[uid], true_item, k))
            ndcgs.append(ndcg_at_k(preds[uid], true_item, k))
        results[f'HR@{k}'] = np.mean(hits) if hits else 0.0
        results[f'NDCG@{k}'] = np.mean(ndcgs) if ndcgs else 0.0
    return results


full_ranking = evaluate_ranking(full_predictions, test_ground_truth, k_values=[1, 5, 10, 20])
print(f'Full-catalog ranking — log-likelihood ({len(full_predictions)} users):')
for m, v in full_ranking.items():
    print(f'  {m}: {v:.4f}')

print('\n--- Cross-paradigm full-catalog HR@10 ---')
print(f'  RAG (200 users):                                {0.0500:.4f}')
print(f'  Generative (log-likelihood, this notebook):    {full_ranking["HR@10"]:.4f}')
print(f'  Generative (title gen, fine-tuned):            {0.0266:.4f}')
print(f'  Explainable BPR-MF:                            {0.0220:.4f}')
print(f'  Generative (zero-shot):                        {0.0050:.4f}')

## 5. Save

In [ ]:
if SMOKE_TEST:
    print('*** SMOKE TEST: skipping save ***')
else:
    results = {
        'paradigm': 'generative_finetuned_full_catalog_likelihood',
        'model': f'QLoRA fine-tuned {MODEL_ID} (full-catalog log-likelihood, KV-cache)',
        'anchor_paper': 'GenRec (Ji et al., ECIR 2024)',
        'ranking_full_catalog': full_ranking,
        'system': {
            'latency': {
                'mean_latency_ms': float(np.mean(latencies) * 1000),
                'p50_latency_ms': float(np.median(latencies) * 1000),
                'p95_latency_ms': float(np.percentile(latencies, 95) * 1000),
            },
            'scoring_method': 'log-likelihood (KV-cache prompt reuse)',
            'catalog_size': len(catalog_titles),
            'top_k_saved': TOP_K_TO_SAVE,
            'n_users_evaluated': len(full_predictions),
        },
    }
    with open(f'{RESULTS_DIR}/generative_full_catalog_likelihood_results.json', 'w') as f:
        json.dump(results, f, indent=2, default=str)
    with open(f'{RESULTS_DIR}/generative_full_catalog_likelihood_predictions.pkl', 'wb') as f:
        pickle.dump(full_predictions, f)
    print(f'Results saved to {RESULTS_DIR}/')